In [2]:
import sklearn
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline
from datetime import datetime, timedelta

In [3]:
df_cctv=pd.read_csv("Test_Dataset/cctv_frames.csv")
df_face=pd.read_csv("Test_Dataset/face_embeddings.csv")
df_lab=pd.read_csv("Test_Dataset/lab_bookings.csv")
df_lib=pd.read_csv("Test_Dataset/library_checkouts.csv")
df_profiles=pd.read_csv("Test_Dataset/student or staff profiles.csv")
df_card=pd.read_csv("Test_Dataset/campus card_swipes.csv")
df_wifi=pd.read_csv("Test_Dataset/wifi_associations_logs.csv")
df_notes=pd.read_csv("Test_Dataset/free_text_notes (helpdesk or RSVPs).csv")

In [4]:
df_notes['text'].count()

np.int64(7000)

In [5]:
df_notes['text'].nunique()

5

In [24]:
import warnings
warnings.filterwarnings('ignore')

TARGET_ENTITY_ID = 'E106124'  # Entity ID to predict
TARGET_DAY_OF_WEEK = 'Monday'  # Day to predict (Monday, Tuesday, etc.)

TIME_INTERVAL_MINUTES = 60  # Time granularity for predictions
START_DATE = '2025-08-25'  # Monday (first date of training period)
END_DATE = '2025-09-27'  # Last date of training period

# Prediction tuning parameters
LOW_ACTIVITY_THRESHOLD = 2  # Users with <= this many logs are considered inactive
HOSTEL_BOOST_FACTOR = 0.3  # Probability boost for hostel if user is inactive
PERSONAL_PATTERN_WEIGHT = 0.6  # Weight for personal history vs similar people

# ============================================================================
# FETCH DEPARTMENT AND ROLE FROM PROFILE
# ============================================================================
print("="*80)
print(f"ENHANCED PERSON TIMELINE PREDICTOR - WEEKLY PATTERN")
print("="*80)
print(f"\nLooking up profile for Entity: {TARGET_ENTITY_ID}")

# Check if entity exists in profiles
if TARGET_ENTITY_ID not in df_profiles['entity_id'].values:
    print(f"ERROR: Entity ID '{TARGET_ENTITY_ID}' not found in df_profiles!")
    print(f"Available entity IDs: {sorted(df_profiles['entity_id'].unique())[:10]}...")
    raise ValueError(f"Invalid TARGET_ENTITY_ID: {TARGET_ENTITY_ID}")

# Get target person's profile and extract department/role
target_profile = df_profiles[df_profiles['entity_id'] == TARGET_ENTITY_ID].iloc[0]
TARGET_DEPARTMENT = target_profile['department']
TARGET_ROLE = target_profile['role']

print(f"✓ Profile found!")
print(f"  Entity ID: {TARGET_ENTITY_ID}")
print(f"  Department: {TARGET_DEPARTMENT}")
print(f"  Role: {TARGET_ROLE}")
print(f"  Day of Week: {TARGET_DAY_OF_WEEK}")
print(f"  Training Period: {START_DATE} to {END_DATE}")
print("="*80)

# ============================================================================
# LOCATION NORMALIZATION MAPPING
# ============================================================================
LOCATION_MAPPING = {
    'AUD': 'AUDITORIUM',
    'AUDITORIUM': 'AUDITORIUM',
    'ADMIN': 'ADMIN_LOBBY',
    'ADMIN_LOBBY': 'ADMIN_LOBBY',
    'CAF': 'CAF',
    'CAF_01': 'CAF',
    'LIB_ENT': 'LIBRARY',
    'LIB': 'LIBRARY',
    'LIBRARY': 'LIBRARY',
    'HOSTEL': 'HOSTEL',
    'HOSTEL_GATE': 'HOSTEL',
    'LAB': 'LAB',
    'LAB_101': 'LAB_101',
    'ENG': 'ENG',
    'LAB_305': 'LAB_305',
    'GYM': 'GYM',
    'SEM_01': 'SEMINAR_ROOM',
    'SEMINAR ROOM': 'SEMINAR_ROOM',
    #'ROOM_A2': 'HOSTEL',
    'LAB_102': 'LAB_102',
    #'ROOM_A1': 'HOSTEL'
}

# Text notes to location mapping
TEXT_TO_LOCATION_MAPPING = {
    'Confirmed attendance for robotics workshop.': 'LAB',
    'Wi-Fi not working in hostel block.': 'HOSTEL',
    'CCTV near library needs check.': 'LIBRARY',
    'Broken chair in seminar room.': 'SEMINAR_ROOM',
    'Requesting lab access.': 'LAB_101',
}

def normalize_location(location):
    """Normalize location names to standard equivalents"""
    if pd.isna(location):
        return location
    location_upper = str(location).upper()
    return LOCATION_MAPPING.get(location_upper, location_upper)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def get_day_of_week(date_str):
    """Convert date string to day of week name"""
    date_obj = datetime.strptime(date_str, '%Y-%m-%d')
    return date_obj.strftime('%A')

def get_all_dates_for_day(start_date, end_date, target_day):
    """Get all dates in range that match the target day of week"""
    dates = []
    current = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')
    
    while current <= end:
        if current.strftime('%A') == target_day:
            dates.append(current.strftime('%Y-%m-%d'))
        current += timedelta(days=1)
    
    return dates

def get_all_dates_in_range(start_date, end_date):
    """Get all dates in range"""
    dates = []
    current = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')
    
    while current <= end:
        dates.append(current.strftime('%Y-%m-%d'))
        current += timedelta(days=1)
    
    return dates

# ============================================================================
# LOAD ALL DATASETS
# ============================================================================
# Silently load datasets without printing intermediate steps

# Get all dates
all_dates = get_all_dates_in_range(START_DATE, END_DATE)
target_dates = get_all_dates_for_day(START_DATE, END_DATE, TARGET_DAY_OF_WEEK)

# Filter profiles for similar people (same department and role)
similar_profiles = df_profiles[
    (df_profiles['department'] == TARGET_DEPARTMENT) & 
    (df_profiles['role'] == TARGET_ROLE)
]

# Create mapping dictionaries
card_to_entity = dict(zip(df_profiles['card_id'], df_profiles['entity_id']))
face_to_entity = dict(zip(df_profiles['face_id'], df_profiles['entity_id']))
hash_to_entity = dict(zip(df_profiles['device_hash'], df_profiles['entity_id']))
entity_to_card = dict(zip(df_profiles['entity_id'], df_profiles['card_id']))
entity_to_face = dict(zip(df_profiles['entity_id'], df_profiles['face_id']))
entity_to_hash = dict(zip(df_profiles['entity_id'], df_profiles['device_hash']))

# ============================================================================
# STEP 2: ANALYZE TARGET PERSON'S COMPLETE HISTORY
# ============================================================================
# Silently analyze complete activity history

all_person_data = []

# Prepare all datasets with timestamps
df_card['timestamp'] = pd.to_datetime(df_card['timestamp'])
df_card['date'] = df_card['timestamp'].dt.strftime('%Y-%m-%d')
df_cctv['timestamp'] = pd.to_datetime(df_cctv['timestamp'])
df_cctv['date'] = df_cctv['timestamp'].dt.strftime('%Y-%m-%d')
df_wifi['timestamp'] = pd.to_datetime(df_wifi['timestamp'])
df_wifi['date'] = df_wifi['timestamp'].dt.strftime('%Y-%m-%d')
df_lab['time_start'] = pd.to_datetime(df_lab['start_time'])
df_lab['time_end'] = pd.to_datetime(df_lab['end_time'])
df_lab['date'] = df_lab['time_start'].dt.strftime('%Y-%m-%d')
df_lib['checkout_time'] = pd.to_datetime(df_lib['timestamp'])
df_lib['date'] = df_lib['checkout_time'].dt.strftime('%Y-%m-%d')
df_notes['timestamp'] = pd.to_datetime(df_notes['timestamp'])
df_notes['date'] = df_notes['timestamp'].dt.strftime('%Y-%m-%d')

# Collect ALL data for target person (across all dates in range)
target_card = entity_to_card.get(TARGET_ENTITY_ID)
target_face = entity_to_face.get(TARGET_ENTITY_ID)
target_hash = entity_to_hash.get(TARGET_ENTITY_ID)

# Card swipes (all dates)
if target_card and target_card in df_card['card_id'].values:
    person_swipes = df_card[
        (df_card['card_id'] == target_card) & 
        (df_card['date'].isin(all_dates))
    ]
    for _, row in person_swipes.iterrows():
        all_person_data.append({
            'time': row['timestamp'],
            'date': row['date'],
            'location': normalize_location(row['location_id']),
            'source': 'Card Swipe'
        })

# CCTV (all dates)
if target_face and target_face in df_cctv['face_id'].values:
    person_cctv = df_cctv[
        (df_cctv['face_id'] == target_face) & 
        (df_cctv['date'].isin(all_dates))
    ]
    for _, row in person_cctv.iterrows():
        all_person_data.append({
            'time': row['timestamp'],
            'date': row['date'],
            'location': normalize_location(row['location_id']),
            'source': 'CCTV'
        })

# WiFi (all dates)
if target_hash and target_hash in df_wifi['device_hash'].values:
    person_wifi = df_wifi[
        (df_wifi['device_hash'] == target_hash) & 
        (df_wifi['date'].isin(all_dates))
    ]
    for _, row in person_wifi.iterrows():
        ap_location = row['ap_id'].split('_')[1] if '_' in row['ap_id'] else 'Unknown'
        all_person_data.append({
            'time': row['timestamp'],
            'date': row['date'],
            'location': normalize_location(ap_location),
            'source': 'WiFi'
        })

# Lab (all dates)
person_lab = df_lab[
    (df_lab['entity_id'] == TARGET_ENTITY_ID) & 
    (df_lab['attended_(YES/NO)'].str.upper() == 'YES') &
    (df_lab['date'].isin(all_dates))
]
for _, row in person_lab.iterrows():
    all_person_data.append({
        'time': row['time_start'],
        'date': row['date'],
        'location': 'LAB',
        'source': 'Lab Attendance'
    })

# Library (all dates)
person_lib = df_lib[
    (df_lib['entity_id'] == TARGET_ENTITY_ID) &
    (df_lib['date'].isin(all_dates))
]
for _, row in person_lib.iterrows():
    all_person_data.append({
        'time': row['checkout_time'],
        'date': row['date'],
        'location': 'LIBRARY',
        'source': 'Library'
    })

# Text Notes (all dates)
person_notes = df_notes[
    (df_notes['entity_id'] == TARGET_ENTITY_ID) &
    (df_notes['date'].isin(all_dates))
]
for _, row in person_notes.iterrows():
    # Extract location from text if mapping exists
    text_content = row['text']
    if text_content in TEXT_TO_LOCATION_MAPPING:
        location = TEXT_TO_LOCATION_MAPPING[text_content]
        all_person_data.append({
            'time': row['timestamp'],
            'date': row['date'],
            'location': location,
            'source': 'Text Notes'
        })

all_person_df = pd.DataFrame(all_person_data)
total_person_logs = len(all_person_df)

# Analyze location patterns
if total_person_logs > 0:
    person_locations = set(all_person_df['location'].unique())
else:
    person_locations = set()

# Check if person is low-activity
is_low_activity = total_person_logs <= LOW_ACTIVITY_THRESHOLD

# Filter for target day of week data
target_day_data = all_person_df[all_person_df['date'].isin(target_dates)]

# ============================================================================
# STEP 3: BUILD P1 - SIMILAR PEOPLE PROBABILITY MODEL
# ============================================================================
# Silently build P1 model

similar_entities = set(similar_profiles['entity_id'])
all_similar_data = []

# Collect data from similar people (only on target day of week)
similar_cards = set(similar_profiles['card_id'])
similar_swipes = df_card[
    (df_card['card_id'].isin(similar_cards)) & 
    (df_card['date'].isin(target_dates))
]
for _, row in similar_swipes.iterrows():
    all_similar_data.append({
        'hour': row['timestamp'].hour,
        'minute_slot': row['timestamp'].minute // TIME_INTERVAL_MINUTES,
        'location': normalize_location(row['location_id'])
    })

similar_faces = set(similar_profiles['face_id'])
similar_cctv = df_cctv[
    (df_cctv['face_id'].isin(similar_faces)) & 
    (df_cctv['date'].isin(target_dates))
]
for _, row in similar_cctv.iterrows():
    all_similar_data.append({
        'hour': row['timestamp'].hour,
        'minute_slot': row['timestamp'].minute // TIME_INTERVAL_MINUTES,
        'location': normalize_location(row['location_id'])
    })

similar_hashes = set(similar_profiles['device_hash'])
similar_wifi = df_wifi[
    (df_wifi['device_hash'].isin(similar_hashes)) & 
    (df_wifi['date'].isin(target_dates))
]
for _, row in similar_wifi.iterrows():
    ap_location = row['ap_id'].split('_')[1] if '_' in row['ap_id'] else 'Unknown'
    all_similar_data.append({
        'hour': row['timestamp'].hour,
        'minute_slot': row['timestamp'].minute // TIME_INTERVAL_MINUTES,
        'location': normalize_location(ap_location)
    })

similar_lab = df_lab[
    (df_lab['entity_id'].isin(similar_entities)) & 
    (df_lab['attended_(YES/NO)'].str.upper() == 'YES') &
    (df_lab['date'].isin(target_dates))
]
for _, row in similar_lab.iterrows():
    all_similar_data.append({
        'hour': row['time_start'].hour,
        'minute_slot': row['time_start'].minute // TIME_INTERVAL_MINUTES,
        'location': 'LAB'
    })

similar_lib = df_lib[
    (df_lib['entity_id'].isin(similar_entities)) &
    (df_lib['date'].isin(target_dates))
]
for _, row in similar_lib.iterrows():
    all_similar_data.append({
        'hour': row['checkout_time'].hour,
        'minute_slot': row['checkout_time'].minute // TIME_INTERVAL_MINUTES,
        'location': 'LIBRARY'
    })

# Text notes from similar people
similar_notes = df_notes[
    (df_notes['entity_id'].isin(similar_entities)) &
    (df_notes['date'].isin(target_dates))
]
for _, row in similar_notes.iterrows():
    text_content = row['text']
    if text_content in TEXT_TO_LOCATION_MAPPING:
        location = TEXT_TO_LOCATION_MAPPING[text_content]
        all_similar_data.append({
            'hour': row['timestamp'].hour,
            'minute_slot': row['timestamp'].minute // TIME_INTERVAL_MINUTES,
            'location': location
        })

similar_df = pd.DataFrame(all_similar_data)

# Build P1 probability matrix
similar_df['time_slot'] = similar_df['hour'].astype(str) + '_' + similar_df['minute_slot'].astype(str)
p1_matrix = similar_df.groupby(['time_slot', 'location']).size().reset_index(name='count')
total_counts = similar_df.groupby('time_slot').size().reset_index(name='total')
p1_matrix = p1_matrix.merge(total_counts, on='time_slot')
p1_matrix['p1_probability'] = p1_matrix['count'] / p1_matrix['total']

# ============================================================================
# STEP 4: BUILD P2 - PERSONAL PATTERN MODEL
# ============================================================================
# Silently build P2 model

# Build personal frequency model for target day
personal_data = []
if len(target_day_data) > 0:
    target_day_data['hour'] = target_day_data['time'].dt.hour
    target_day_data['minute_slot'] = target_day_data['time'].dt.minute // TIME_INTERVAL_MINUTES
    
    for _, row in target_day_data.iterrows():
        personal_data.append({
            'hour': row['hour'],
            'minute_slot': row['minute_slot'],
            'location': row['location']
        })

personal_df = pd.DataFrame(personal_data)

if len(personal_df) > 0:
    personal_df['time_slot'] = personal_df['hour'].astype(str) + '_' + personal_df['minute_slot'].astype(str)
    p2_matrix = personal_df.groupby(['time_slot', 'location']).size().reset_index(name='personal_count')
    personal_total = personal_df.groupby('time_slot').size().reset_index(name='personal_total')
    p2_matrix = p2_matrix.merge(personal_total, on='time_slot')
    p2_matrix['p2_probability'] = p2_matrix['personal_count'] / p2_matrix['personal_total']
else:
    p2_matrix = pd.DataFrame(columns=['time_slot', 'location', 'personal_count', 'personal_total', 'p2_probability'])

# ============================================================================
# STEP 5: COMBINE P1 AND P2 WITH CONSTRAINTS
# ============================================================================
# Silently combine models with constraints

# Get all unique locations from similar people
all_locations = set(p1_matrix['location'].unique())

# Get all time slots
all_time_slots = sorted(p1_matrix['time_slot'].unique(), 
                       key=lambda x: (int(x.split('_')[0]), int(x.split('_')[1])))

# Build combined probability matrix
combined_predictions = []

for time_slot in all_time_slots:
    hour, slot = map(int, time_slot.split('_'))
    
    # Get P1 probabilities for this time slot
    p1_data = p1_matrix[p1_matrix['time_slot'] == time_slot].copy()
    
    # Get P2 probabilities if available
    p2_data = p2_matrix[p2_matrix['time_slot'] == time_slot].copy() if len(p2_matrix) > 0 else pd.DataFrame()
    
    # For each location in P1
    for _, p1_row in p1_data.iterrows():
        location = p1_row['location']
        p1_prob = p1_row['p1_probability']
        
        # CONSTRAINT 1: Never visited locations get 0 probability
        if person_locations and location not in person_locations:
            # This person has data but never went to this location
            final_prob = 0.0
            method = "EXCLUDED (Never visited)"
            combined_predictions.append({
                'time_slot': time_slot,
                'hour': hour,
                'minute_slot': slot,
                'location': location,
                'p1_prob': p1_prob,
                'p2_prob': 0.0,
                'final_probability': final_prob,
                'method': method
            })
            continue
        
        # Get P2 probability if exists
        p2_prob = 0.0
        if len(p2_data) > 0:
            p2_match = p2_data[p2_data['location'] == location]
            if len(p2_match) > 0:
                p2_prob = p2_match['p2_probability'].values[0]
        
        # Combine P1 and P2
        if p2_prob > 0:
            # We have personal data - weighted combination
            final_prob = (PERSONAL_PATTERN_WEIGHT * p2_prob + 
                         (1 - PERSONAL_PATTERN_WEIGHT) * p1_prob)
            method = f"P1+P2 (weights: {1-PERSONAL_PATTERN_WEIGHT:.1f}/{PERSONAL_PATTERN_WEIGHT:.1f})"
        else:
            # No personal data - use P1
            final_prob = p1_prob
            method = "P1 only"
        
        # CONSTRAINT 2: Low activity boost for HOSTEL
        if is_low_activity and 'HOSTEL' in location.upper():
            final_prob = min(1.0, final_prob + HOSTEL_BOOST_FACTOR)
            method = f"{method} + HOSTEL BOOST"
        
        combined_predictions.append({
            'time_slot': time_slot,
            'hour': hour,
            'minute_slot': slot,
            'location': location,
            'p1_prob': p1_prob,
            'p2_prob': p2_prob,
            'final_probability': final_prob,
            'method': method
        })

combined_df = pd.DataFrame(combined_predictions)

# Normalize probabilities per time slot
for time_slot in all_time_slots:
    mask = combined_df['time_slot'] == time_slot
    total_prob = combined_df.loc[mask, 'final_probability'].sum()
    if total_prob > 0:
        combined_df.loc[mask, 'final_probability'] = combined_df.loc[mask, 'final_probability'] / total_prob

# ============================================================================
# STEP 6: GENERATE FINAL TIMELINE
# ============================================================================
# Silently generate final timeline

timeline = []

for hour in range(24):
    for slot in range(60 // TIME_INTERVAL_MINUTES):
        time_slot = f"{hour}_{slot}"
        time_str = f"{hour:02d}:{slot*TIME_INTERVAL_MINUTES:02d}"
        
        # Get predictions for this time slot
        slot_predictions = combined_df[combined_df['time_slot'] == time_slot]
        
        if len(slot_predictions) > 0:
            # Get best prediction
            best = slot_predictions.loc[slot_predictions['final_probability'].idxmax()]
            
            # Get top 3 alternatives
            top3 = slot_predictions.nlargest(3, 'final_probability')
            alternatives = ', '.join([
                f"{row['location']}({row['final_probability']:.2%})" 
                for _, row in top3.iterrows()
            ])
            
            timeline.append({
                'time': time_str,
                'location': best['location'],
                'probability': best['final_probability'],
                'p1_prob': best['p1_prob'],
                'p2_prob': best['p2_prob'],
                'method': best['method'],
                'alternatives': alternatives
            })
        else:
            timeline.append({
                'time': time_str,
                'location': 'Unknown',
                'probability': 0.0,
                'p1_prob': 0.0,
                'p2_prob': 0.0,
                'method': 'No Data',
                'alternatives': 'None'
            })

timeline_df = pd.DataFrame(timeline)

# ============================================================================
# DISPLAY RESULTS
# ============================================================================
print("\n" + "="*80)
print(f"FINAL {TARGET_DAY_OF_WEEK.upper()} TIMELINE FOR {TARGET_ENTITY_ID}")
print("="*80)
display(timeline_df[['time', 'location', 'probability']])

print("\n" + "="*80)
print("ANALYSIS SUMMARY")
print("="*80)
print(f"Total activity logs (all dates): {total_person_logs}")
print(f"Activity logs on {TARGET_DAY_OF_WEEK}s: {len(target_day_data)}")
print(f"Low activity user: {is_low_activity}")
print(f"Locations ever visited: {sorted(person_locations) if person_locations else 'None'}")
print(f"\nPrediction breakdown:")
print(timeline_df['method'].value_counts())

# High confidence predictions
high_confidence = timeline_df[timeline_df['probability'] >= 0.5]
print(f"\nHigh confidence predictions (≥50%): {len(high_confidence)} time slots")
if len(high_confidence) > 0:
    display(high_confidence[['time', 'location', 'probability', 'method']])

# Show excluded locations
if person_locations:
    excluded_locations = all_locations - person_locations
    if excluded_locations:
        print(f"\nLocations EXCLUDED (never visited by user): {sorted(excluded_locations)}")

ENHANCED PERSON TIMELINE PREDICTOR - WEEKLY PATTERN

Looking up profile for Entity: E106124
✓ Profile found!
  Entity ID: E106124
  Department: Maths
  Role: student
  Day of Week: Monday
  Training Period: 2025-08-25 to 2025-09-27

FINAL MONDAY TIMELINE FOR E106124


,time,location,probability
0,00:00,LAB_305,0.519231
1,01:00,LAB_101,0.368421
2,02:00,LIBRARY,0.333333
3,03:00,LIBRARY,0.416667
4,04:00,LAB_305,0.300000
5,05:00,LIBRARY,0.296296
6,06:00,AUDITORIUM,0.217391
7,07:00,LIBRARY,0.375000
8,08:00,LIBRARY,0.461538
9,09:00,LAB,0.291667



ANALYSIS SUMMARY
Total activity logs (all dates): 7
Activity logs on Mondays: 2
Low activity user: False
Locations ever visited: ['AUDITORIUM', 'LAB', 'LAB_101', 'LAB_305', 'LIBRARY']

Prediction breakdown:
method
P1 only                     22
P1+P2 (weights: 0.4/0.6)     2
Name: count, dtype: int64

High confidence predictions (≥50%): 3 time slots


,time,location,probability,method
0,00:00,LAB_305,0.519231,P1+P2 (weights: 0.4/0.6)
16,16:00,LIBRARY,0.590909,P1 only
22,22:00,LAB,0.604651,P1+P2 (weights: 0.4/0.6)



Locations EXCLUDED (never visited by user): ['ADMIN_LOBBY', 'CAF', 'ENG', 'GYM', 'HOSTEL', 'SEMINAR_ROOM']


In [23]:
# ============================================================================
# SECURITY ALERT SYSTEM
# ============================================================================
# This system analyzes the timeline_df generated from the prediction system
# and raises alerts for extended periods of low confidence predictions

def check_security_alerts(timeline_df, threshold=0.5, consecutive_hours=8):
    """
    Checks for consecutive periods where prediction probability is low.
    
    Parameters:
    -----------
    timeline_df : DataFrame
        The timeline dataframe with columns: time, location, probability, method
    threshold : float
        Probability threshold below which we consider it "low confidence" (default: 0.5)
    consecutive_hours : int
        Number of consecutive hours of low confidence that trigger an alert (default: 12)
    
    Returns:
    --------
    alerts : list of dict
        List of alert periods with start time, end time, and duration
    """
    
    print("="*80)
    print("SECURITY ALERT SYSTEM")
    print("="*80)
    print(f"Monitoring for periods with probability ≤ {threshold:.1%}")
    print(f"Alert threshold: {consecutive_hours} consecutive hours")
    print("="*80)
    
    # Ensure timeline is sorted by time
    timeline_df = timeline_df.sort_values('time').reset_index(drop=True)
    
    # Track low confidence periods
    alerts = []
    current_low_period_start = None
    current_low_period_count = 0
    
    for idx, row in timeline_df.iterrows():
        is_low_confidence = row['probability'] <= threshold
        
        if is_low_confidence:
            # Start or continue a low confidence period
            if current_low_period_start is None:
                current_low_period_start = row['time']
                current_low_period_count = 1
            else:
                current_low_period_count += 1
        else:
            # End of low confidence period - check if it meets alert criteria
            if current_low_period_start is not None:
                if current_low_period_count >= consecutive_hours:
                    alerts.append({
                        'start_time': current_low_period_start,
                        'end_time': timeline_df.iloc[idx-1]['time'],
                        'duration_hours': current_low_period_count,
                        'severity': 'HIGH' if current_low_period_count >= 18 else 'MEDIUM'
                    })
                
                # Reset tracking
                current_low_period_start = None
                current_low_period_count = 0
    
    # Check if we're still in a low confidence period at the end
    if current_low_period_start is not None:
        if current_low_period_count >= consecutive_hours:
            alerts.append({
                'start_time': current_low_period_start,
                'end_time': timeline_df.iloc[-1]['time'],
                'duration_hours': current_low_period_count,
                'severity': 'HIGH' if current_low_period_count >= 18 else 'MEDIUM'
            })
    
    return alerts


def display_security_report(timeline_df, alerts, entity_id, day_of_week):
    """
    Display comprehensive security alert report.
    """
    
    print(f"\n{'='*80}")
    print(f"SECURITY REPORT FOR {entity_id} - {day_of_week.upper()}")
    print(f"{'='*80}")
    
    # Overall statistics
    total_slots = len(timeline_df)
    low_conf_slots = len(timeline_df[timeline_df['probability'] <= 0.5])
    avg_probability = timeline_df['probability'].mean()
    min_probability = timeline_df['probability'].min()
    
    print(f"\n📊 Overall Statistics:")
    print(f"   Total time slots: {total_slots}")
    print(f"   Low confidence slots (≤50%): {low_conf_slots} ({low_conf_slots/total_slots*100:.1f}%)")
    print(f"   Average probability: {avg_probability:.2%}")
    print(f"   Minimum probability: {min_probability:.2%}")
    
    # Alert summary
    if alerts:
        print(f"\n🚨 SECURITY ALERTS: {len(alerts)} PERIOD(S) DETECTED")
        print(f"{'='*80}")
        
        for i, alert in enumerate(alerts, 1):
            severity_icon = "🔴" if alert['severity'] == 'HIGH' else "🟠"
            print(f"\n{severity_icon} Alert #{i} - {alert['severity']} PRIORITY")
            print(f"   Start Time: {alert['start_time']}")
            print(f"   End Time: {alert['end_time']}")
            print(f"   Duration: {alert['duration_hours']} hours")
            print(f"   Status: LOW CONFIDENCE TRACKING")
            
            # Get the locations during this period
            period_data = timeline_df[
                (timeline_df['time'] >= alert['start_time']) & 
                (timeline_df['time'] <= alert['end_time'])
            ]
            
            print(f"\n   Predicted locations during this period:")
            location_summary = period_data.groupby('location').agg({
                'probability': ['mean', 'count']
            }).round(3)
            location_summary.columns = ['Avg Probability', 'Time Slots']
            location_summary = location_summary.sort_values('Time Slots', ascending=False)
            print(location_summary.to_string(header=True, index=True))
            
            print(f"\n   Recommendation: Enhanced monitoring required during {alert['start_time']}-{alert['end_time']}")
            print(f"   " + "-"*70)
        
        # Critical recommendations
        print(f"\n{'='*80}")
        print("⚠️  RECOMMENDED ACTIONS:")
        print(f"{'='*80}")
        total_alert_hours = sum(a['duration_hours'] for a in alerts)
        print(f"1. Total time under low confidence: {total_alert_hours} hours ({total_alert_hours/24*100:.1f}% of day)")
        print(f"2. Deploy additional surveillance during identified periods")
        print(f"3. Cross-reference with other security data sources")
        print(f"4. Consider gathering more training data for this entity/day")
        
        high_priority_alerts = [a for a in alerts if a['severity'] == 'HIGH']
        if high_priority_alerts:
            print(f"5. URGENT: {len(high_priority_alerts)} HIGH priority period(s) require immediate attention")
        
    else:
        print(f"\n✅ NO SECURITY ALERTS DETECTED")
        print(f"   All time periods have acceptable confidence levels (>50%)")
        print(f"   Tracking confidence: GOOD")
    
    print(f"\n{'='*80}")


def visualize_confidence_timeline(timeline_df):
    """
    Create a visual representation of confidence levels throughout the day.
    """
    print(f"\n📈 CONFIDENCE TIMELINE (24 HOURS)")
    print(f"{'='*80}")
    
    # Group by hour for cleaner visualization
    timeline_df['hour'] = timeline_df['time'].str.split(':').str[0].astype(int)
    hourly_conf = timeline_df.groupby('hour')['probability'].mean()
    
    for hour in range(24):
        if hour in hourly_conf.index:
            prob = hourly_conf[hour]
            bar_length = int(prob * 40)  # Scale to 40 characters max
            
            # Color coding based on confidence
            if prob > 0.7:
                marker = "█"
                status = "HIGH"
            elif prob > 0.5:
                marker = "▓"
                status = "MED "
            else:
                marker = "░"
                status = "LOW "
            
            bar = marker * bar_length
            print(f"{hour:02d}:00 [{status}] {bar} {prob:.2%}")
        else:
            print(f"{hour:02d}:00 [----] No data")
    
    print(f"{'='*80}")
    print("Legend: █ = High confidence (>70%)  ▓ = Medium (>50%)  ░ = Low (≤50%)")


# ============================================================================
# MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    # Assuming timeline_df exists from the prediction script
    # If running standalone, you need to load or pass timeline_df
    
    try:
        # Check if timeline_df exists
        if 'timeline_df' not in locals() and 'timeline_df' not in globals():
            print("ERROR: timeline_df not found!")
            print("Please run the prediction script first to generate timeline_df")
        else:
            # Run security checks
            alerts = check_security_alerts(
                timeline_df, 
                threshold=0.5, 
                consecutive_hours=12
            )
            
            # Display comprehensive report
            display_security_report(
                timeline_df, 
                alerts, 
                TARGET_ENTITY_ID if 'TARGET_ENTITY_ID' in globals() else 'Unknown',
                TARGET_DAY_OF_WEEK if 'TARGET_DAY_OF_WEEK' in globals() else 'Unknown'
            )
            
            # Visualize confidence timeline
            visualize_confidence_timeline(timeline_df)
            
            # Export alerts if any
            if alerts:
                alerts_df = pd.DataFrame(alerts)
                print(f"\n💾 Alert summary:")
                print(alerts_df.to_string(index=False))
                
                # Optionally save to CSV
                # alerts_df.to_csv(f'security_alerts_{TARGET_ENTITY_ID}_{TARGET_DAY_OF_WEEK}.csv', index=False)
                # print(f"\nAlerts exported to CSV")
    
    except NameError as e:
        print(f"ERROR: Required variables not found - {e}")
        print("Make sure to run the prediction script first!")

SECURITY ALERT SYSTEM
Monitoring for periods with probability ≤ 50.0%
Alert threshold: 12 consecutive hours

SECURITY REPORT FOR E100300 - MONDAY

📊 Overall Statistics:
   Total time slots: 24
   Low confidence slots (≤50%): 0 (0.0%)
   Average probability: 89.60%
   Minimum probability: 71.43%

✅ NO SECURITY ALERTS DETECTED
   All time periods have acceptable confidence levels (>50%)
   Tracking confidence: GOOD


📈 CONFIDENCE TIMELINE (24 HOURS)
00:00 [HIGH] ████████████████████████████████████████ 100.00%
01:00 [HIGH] ████████████████████████████████████████ 100.00%
02:00 [HIGH] ███████████████████████████████████ 88.89%
03:00 [HIGH] ████████████████████████████████████████ 100.00%
04:00 [HIGH] ███████████████████████████████ 80.00%
05:00 [HIGH] ████████████████████████████████ 81.82%
06:00 [HIGH] ████████████████████████████████ 80.00%
07:00 [HIGH] ████████████████████████████████████████ 100.00%
08:00 [HIGH] ████████████████████████████████████████ 100.00%
09:00 [HIGH] ███████████